In [10]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
# Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset, DatasetDict

raw_dataset = load_dataset('json', data_files='../datasets/article_summary/valid_finnish_articles_200.json')


original_train = raw_dataset['train']

# split the dataset into train and validation sets
train_valid = original_train.train_test_split(test_size=0.2, seed=42)

# create a DatasetDict object
dataset = DatasetDict({
    'train': train_valid['train'],
    'validation': train_valid['test']
})

In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 103
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 26
    })
})


In [12]:
# Load tokenizer and model
model_name = "google/mt5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Preprocessing function
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        examples["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [13]:
# Tokenize
tokenized_dataset = dataset.map(preprocess_function, batched=True)

training_args = Seq2SeqTrainingArguments(
    eval_strategy="epoch",
    learning_rate=5e-5,
    logging_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=20,
    predict_with_generate=True,
    fp16=False,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
)

trainer.train()

trainer.save_model("./finetuned-mt5-finnish-summarizer")

Map: 100%|██████████| 26/26 [00:00<00:00, 1189.60 examples/s]
/tmp/ipykernel_81884/2569167106.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,28.470400,26.363150
2,23.534000,21.961834
3,19.970000,18.447460
4,17.387100,15.610986
5,14.840700,12.730300
6,12.175900,8.156831
7,9.337600,5.913432
8,7.368000,5.126870
9,6.278100,4.187683
10,5.674700,3.734904


In [14]:
from projects.summarizer.datascrape import scrape_article
from transformers import pipeline

summarizer = pipeline("text2text-generation", model="./finetuned-mt5-finnish-summarizer", tokenizer="google/mt5-small")

new_text = scrape_article("https://yle.fi/a/74-20158531", "yle")

input_text = "Yhteenveto: " + new_text

summary = summarizer(input_text, max_length=100, min_length=50, do_sample=False)
print(summary[0])

/home/stefu/anaconda3/envs/finalproject/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


{'generated_text': '<extra_id_0>a tunke ja a tunk ja n tunkeutumistapausta. . . . . . . . . . . . . . . .'}
